# Indiana Bosonic Code - DMANH xSDF npy angles


In [5]:
from jaqalpaq import run
from jaqalpaq import emulator
from jaqalpaq.run import run_jaqal_file, run_jaqal_string, run_jaqal_batch, run_jaqal_circuit, frontend

from jaqalpaq.emulator.unitary import UnitarySerializedEmulator
emulator_backend = UnitarySerializedEmulator()

#from Experiment import Experiment
import numpy as np
import csv
from datetime import datetime, date, time, timezone
from pytz import timezone
from collections import OrderedDict
import pytz
from scipy.special import jn_zeros
import math
import matplotlib.pyplot as plt
import pickle as pkl
from jaqalpaq.parser import parse_jaqal_string
from pprint import pprint

def timestamp_generate():
    mountain = timezone('US/Mountain')
    timestamp_utc = datetime.utcnow()
    timestamp_local = timestamp_utc.astimezone(mountain)
    return(timestamp_utc, timestamp_local)

def int_to_base(n, b):
    if n == 0:
        return "0"
    digits = []
    is_negative = n < 0
    n = abs(n)
    while n:
        digits.append(int(n % b))
        n //= b
    if is_negative:
        digits.append("-")
    return ''.join(str(x) if x < 10 else chr(55 + x) for x in reversed(digits))



In [12]:
# Can import a file here with the angles in it, so angles can be called automatically.

nq = 2
repeats = 1000
im_beta_list = np.linspace(-0.4, 0.4, 5).tolist()
override_dict = {"__repeats__": repeats,
                 "imBeta": im_beta_list}

# Keep the readout beta sweep and SQR/Rz angle explicit in this notebook.
# The compiler export supplies the per-step SDF displacement angles.
Z_ANGLE = -0.8
from pathlib import Path

# You may need to adjust the path of the angles file. - JBG
angles_path = Path('../build/dmanh_angles.npy')
compiled_angles = np.load(angles_path)  # rows: first xSDF re, first xSDF im, phi - JBG

if compiled_angles.shape[0] != 3:
    raise ValueError(f"expected angles.npy with shape (3, steps), got {compiled_angles.shape}")

angle = (-compiled_angles[:2].T).tolist()
num_subcircuits = len(angle) + 1  # subcircuits with 0, 1, ..., N Trotter steps
subcircuit_steps = np.arange(num_subcircuits)


In [13]:
class JaqalGenerator:
    Z_angle = Z_ANGLE
    header = (
        "from Calibration_PulseDefinitions.QubitBosonPulses usepulses *\n"
        f"let phi {Z_angle}\n"
        "let reBeta 0\n"
        "let imBeta 0\n\n"

        "register q[2]\n\n"
    )

    subcircuit_header = (
        "// ******************* //\n"
        "// Subcircuit with {subcirc_idx} Trotter steps\n"
        "prepare_all\n"
        "// Wavepacket preparation: |down>|0> -> |down>|x=-x_min> using zCD.\n"
        "zSDF q[0] 0.89021208217480396 0\n\n"
    )

    trotter_step = (
        "// Evolution step {time_step} \n"
        "xSDF q[0] {m_reBeta} {m_imBeta}\n"
        "Rz q[0] phi\n"
        "xSDF q[0] {reBeta} {imBeta}\n"
        "xSDF q[0] {reBeta} {imBeta}\n"
        "Rz q[0] phi\n"
        "xSDF q[0] {m_reBeta} {m_imBeta}\n"
    )
    
    subcircuit_footer = (
        "// Characteristic-function readout.\n"
        "R q[1] 0 1.5707963267948966\n"
        "xSDF q[1] reBeta imBeta\n"
        "measure_all\n"
        "// ******************* //\n\n\n"
    )

    @classmethod
    def generate(self, angle_data: list[list[float]]) -> str:
        trotter_steps = len(angle_data)
        jaqal_string = self.header
        for ii in range(trotter_steps + 1):
            jaqal_string += self._gen_subcircuit(ii, angle_data)
        return jaqal_string
    
    @classmethod
    def _gen_subcircuit(self, trotter_steps: int, angle_data: list[list[float]]) -> str:
        subcircuit = self.subcircuit_header.format(subcirc_idx=trotter_steps)
        for ii in range(trotter_steps):
            subcircuit += self.trotter_step.format(time_step=ii + 1,
                                                   reBeta=angle_data[ii][0],
                                                   imBeta=angle_data[ii][1],
                                                   m_reBeta=-angle_data[ii][0],
                                                   m_imBeta=-angle_data[ii][1])
        subcircuit += self.subcircuit_footer
        return subcircuit


In [14]:
# The for loop was removed because batching has been moved to overrides (also some of the names have changed, sorry).

jaqal_string = JaqalGenerator.generate(angle_data=angle)

print(jaqal_string)

from Calibration_PulseDefinitions.QubitBosonPulses usepulses *
let phi -0.8
let reBeta 0
let imBeta 0

register q[2]

// ******************* //
// Subcircuit with 0 Trotter steps
prepare_all
// Wavepacket preparation: |down>|0> -> |down>|x=-x_min> using zCD.
zSDF q[0] 0.89021208217480396 0

// Characteristic-function readout.
R q[1] 0 1.5707963267948966
xSDF q[1] reBeta imBeta
measure_all
// ******************* //


// ******************* //
// Subcircuit with 1 Trotter steps
prepare_all
// Wavepacket preparation: |down>|0> -> |down>|x=-x_min> using zCD.
zSDF q[0] 0.89021208217480396 0

// Evolution step 1 
xSDF q[0] -0.18512012242326523 -0.0
Rz q[0] phi
xSDF q[0] 0.18512012242326523 0.0
xSDF q[0] 0.18512012242326523 0.0
Rz q[0] phi
xSDF q[0] -0.18512012242326523 -0.0
// Characteristic-function readout.
R q[1] 0 1.5707963267948966
xSDF q[1] reBeta imBeta
measure_all
// ******************* //


// ******************* //
// Subcircuit with 2 Trotter steps
prepare_all
// Wavepacket prepar

In [ ]:
results = run.run_jaqal_string(jaqal_string, overrides = override_dict)

# print(results)

In [ ]:
def exp_z(prob): #Postselect the results on the 
    """Expectation value of Z"""
    #probe qubit's state probabilities
    if prob[0] + prob[2] == 0:
        state0 = 0
        state1 = 0
    else:
        state0 = prob[0]/(prob[0]+prob[2])
        state1 = prob[2]/(prob[0]+prob[2])
    
    return state0 - state1

In [ ]:
# Extract the data from the results object.
# This extracts the axis data.

imMeas = np.empty((len(im_beta_list), num_subcircuits, 2 ** nq))
expZ_imMeas = np.empty((len(im_beta_list), num_subcircuits))

print(len(results.by_subbatch[0].by_subcircuit[0].probability_by_int[:,0]))

for ii in range(len(im_beta_list)):
    for jj in range(num_subcircuits):
        imMeas[ii, jj, :] = results.by_subbatch[ii].by_subcircuit[jj].probability_by_int[:,0]
        expZ_imMeas[ii, jj] = exp_z(imMeas[ii, jj])
print(imMeas[0, 0])
print(expZ_imMeas)
np.save('expZ_imMeas', expZ_imMeas) 

# If you want to have the outputs available in a text format, the save functions below are useful.
# # Save all probabilities
# np.save('prob_imMeas', imMeas)

# # Save beta ranges
# np.save('ImBetas', im_values)


In [ ]:
fig, axs = plt.subplots(3,2, figsize=(14, 14))
fig.subplots_adjust(hspace = .15, wspace=.1)        
axes = axs.ravel()

for ii, beta in enumerate(im_beta_list):
    axes[ii].plot(subcircuit_steps, expZ_imMeas[ii, :], label=f"{beta:.2f}")
    axes[ii].set_title(f"Im[beta] = {beta:.3f}")
    axes[ii].set_xlabel("Number of trotter steps")
    axes[ii].set_ylabel("Im[chi]")
fig.tight_layout()

lastplot = len(im_beta_list)

for  ii, beta in enumerate(im_beta_list):
    axes[lastplot].plot(subcircuit_steps, expZ_imMeas[ii, :], label=f"{beta:.2f}")
axes[lastplot].set_title('All Im[beta]')
axes[lastplot].set_xlabel("Number of trotter steps")
axes[lastplot].set_ylabel("Im[chi]")
axes[lastplot].legend(title="Im[beta]")
fig.tight_layout()
